# NevoScan

## Эксперимент 6: Многозадачная архитектура (8 голов)

**Архитектура:** EfficientNet-B3 + 8 независимых классификационных голов    

| Голова | Признак | Балл Argenziano | Вес в лоссе |
|---|---|---|---|
| 1 | Пигментная сеть | 2 | 1.5 |
| 2 | Полосы | 1 | 1.0 |
| 3 | Пигментация | 1 | 1.0 |
| 4 | Регрессия | 1 | 1.0 |
| 5 | Точки/Глобулы | 1 | 1.0 |
| 6 | Бело-голубая вуаль | 2 | 1.5 |
| 7 | Сосудистые структуры | 2 | 1.5 |
| 8 | Диагноз (меланома/нет) | — | 2.0 |



**Данные:** Derm7pt (train=413) + ISIC Task 2 (2594) = 3007 изображений

**Вход:** 4 канала (RGB + сводная маска), стратегия A из Эксп.5  

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

### 1. Монтируем Google Drive и данные

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

os.makedirs('/content/dataset', exist_ok=True)
if not os.path.exists('/content/dataset/release_v0'):
    os.system('unzip -q "/content/drive/MyDrive/Диплом/практика_преддипломная/derm7pt.zip" '
              '-d "/content/dataset"')
    print('✓ Derm7pt распакован')
else:
    print('✓ Derm7pt уже есть')

os.makedirs('/content/isic', exist_ok=True)
if not os.path.exists('/content/isic/images'):
    os.system('wget -q --show-progress -O /content/isic/images.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task1-2_Training_Input.zip')
    os.system('unzip -q /content/isic/images.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task1-2_Training_Input" /content/isic/images')
    os.system('rm -rf /content/isic/images.zip /content/isic/tmp')
    print('✓ ISIC images готовы')
else:
    print('✓ ISIC images уже есть')

if not os.path.exists('/content/isic/masks'):
    os.system('wget -q --show-progress -O /content/isic/masks.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task2_Training_GroundTruth_v3.zip')
    os.system('unzip -q /content/isic/masks.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task2_Training_GroundTruth_v3" /content/isic/masks')
    os.system('rm -rf /content/isic/masks.zip /content/isic/tmp')
    print('✓ ISIC masks готовы')
else:
    print('✓ ISIC masks уже есть')

### 2. Импорты и конфигурация

Гиперпараметры идентичны Эксп.5A. Добавлены веса голов для многозадачного обучения

In [ ]:
import glob, json, shutil
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score,
    recall_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Пути
DERM_BASE  = '/content/dataset/release_v0'
DERM_IMG   = os.path.join(DERM_BASE, 'images')
DERM_META  = os.path.join(DERM_BASE, 'meta/meta.csv')
DERM_TRAIN = os.path.join(DERM_BASE, 'meta/train_indexes.csv')
DERM_VAL   = os.path.join(DERM_BASE, 'meta/valid_indexes.csv')
DERM_TEST  = os.path.join(DERM_BASE, 'meta/test_indexes.csv')
ISIC_IMG   = '/content/isic/images'
ISIC_MASKS = '/content/isic/masks'
SAVE_DIR   = '/content/drive/MyDrive/Диплом/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Гиперпараметры
IMG_SIZE     = 300
BATCH_SIZE   = 16
NUM_EPOCHS   = 12
LR           = 1e-4
WEIGHT_DECAY = 1e-3
THRESHOLD    = 0.4
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.0
FN_WEIGHT    = 3.0

# Признаки (порядок = индексы голов 0-6)
FEATURES = [
    'pigment_network', 'streaks', 'pigmentation',
    'regression_structures', 'dots_and_globules',
    'blue_whitish_veil', 'vascular_structures'
]
FEAT_RU = [
    'Пигм. сеть', 'Полосы', 'Пигментация',
    'Регрессия', 'Точки/Глобулы', 'Бело-гол. вуаль', 'Сос. структуры'
]

# Веса по шкале Argenziano
ARGENZIANO_WEIGHTS = {
    'pigment_network': 2, 'streaks': 1, 'pigmentation': 1,
    'regression_structures': 1, 'dots_and_globules': 1,
    'blue_whitish_veil': 2, 'vascular_structures': 2,
}
ARGENZIANO_SCORES   = [ARGENZIANO_WEIGHTS[f] for f in FEATURES]
SUSPICION_THRESHOLD = 3

# Веса голов в многозадачном лоссе:
# большие критерии Argenziano (2б): 1.5, малые (1б): 1.0, диагноз: 2.0
HEAD_WEIGHTS = [1.5, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 2.0]

# Соответствие признаков и масок ISIC Task 2
ISIC_ATTR = {
    'pigment_network':       'pigment_network',
    'streaks':               'streaks',
    'dots_and_globules':     'globules',
    'pigmentation':          None,
    'regression_structures': None,
    'blue_whitish_veil':     None,
    'vascular_structures':   None,
}

print('✓ Конфиг загружен')
print(f'  Веса голов: {dict(zip(FEAT_RU + ["Диагноз"], HEAD_WEIGHTS))}')

### 3. Загрузка данных

Ключевое отличие от Эксп.5: добавляем колонку diagnosis_bin из Derm7pt для 8-й головы.  

Меланома = 1, всё остальное = 0. Для ISIC метка диагноза = −1 (если нет аннотации, то игнорируем).

In [ ]:
def map_label(text):
    if pd.isna(text): return 0
    return 0 if str(text).lower().strip() in ['absent', 'regular', 'typical'] else 1

def map_diagnosis(text):
    """Меланома = 1, всё остальное = 0."""
    if pd.isna(text): return 0
    return 1 if 'melanoma' in str(text).lower() else 0

# Derm7pt
df_meta = pd.read_csv(DERM_META)
print(f'Колонки meta.csv: {list(df_meta.columns)}')

df_derm = df_meta.copy()
for feat in FEATURES:
    df_derm[feat] = df_derm[feat].apply(map_label)

# Диагностическая метка для 8-й головы
if 'diagnosis' in df_derm.columns:
    df_derm['diagnosis_bin'] = df_derm['diagnosis'].apply(map_diagnosis)
    n_mel = df_derm['diagnosis_bin'].sum()
    print(f'\nДиагнозы в Derm7pt:')
    print(df_meta['diagnosis'].value_counts().to_string())
    print(f'\nМеланом (diagnosis_bin=1): {n_mel} ({100*n_mel/len(df_derm):.1f}%)')
else:
    print(' Колонка diagnosis не найдена — 8-я голова будет обучаться с меткой -1')
    df_derm['diagnosis_bin'] = -1

# Пути к файлам
all_files  = glob.glob(os.path.join(DERM_IMG, '**/*'), recursive=True)
path_map   = {f.lower(): f for f in all_files if os.path.isfile(f)}
df_derm['full_path']    = df_derm['derm'].apply(
    lambda x: path_map.get(os.path.join(DERM_IMG, x).lower()))
df_derm['mask_path']   = None
df_derm['source']      = 'derm7pt'
df_derm = df_derm[df_derm['full_path'].notna()].reset_index(drop=True)

# Официальные сплиты
train_idx  = pd.read_csv(DERM_TRAIN).iloc[:, 0].values
val_idx    = pd.read_csv(DERM_VAL).iloc[:, 0].values
test_idx   = pd.read_csv(DERM_TEST).iloc[:, 0].values
derm_train = df_derm.iloc[train_idx].reset_index(drop=True)
val_df     = df_derm.iloc[val_idx].reset_index(drop=True)
test_df    = df_derm.iloc[test_idx].reset_index(drop=True)
print(f'\nDerm7pt — train: {len(derm_train)} | val: {len(val_df)} | test: {len(test_df)}')

# Меланомы в train
if 'diagnosis_bin' in derm_train.columns:
    n = derm_train['diagnosis_bin'].sum()
    print(f'Меланом в train: {n} ({100*n/len(derm_train):.1f}%)')

# ISIC Task 2
os.makedirs('/content/isic/combined_masks', exist_ok=True)

def get_isic_label(img_id, feat):
    attr = ISIC_ATTR.get(feat)
    if attr is None: return -1
    mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
    if not os.path.exists(mf): return 0
    return 1 if np.array(Image.open(mf).convert('L')).max() > 0 else 0

def build_combined_mask(img_id):
    combined = None
    for attr in ['pigment_network', 'negative_network', 'streaks',
                 'milia_like_cysts', 'globules']:
        mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
        if os.path.exists(mf):
            m = np.array(Image.open(mf).convert('L'))
            combined = m if combined is None else np.maximum(combined, m)
    if combined is None: return None
    out = f'/content/isic/combined_masks/{img_id}_combined.png'
    Image.fromarray(combined).save(out)
    return out

isic_files = sorted([f for f in os.listdir(ISIC_IMG) if f.endswith('.jpg')])
records = []
for fname in tqdm(isic_files, desc='Парсим ISIC'):
    img_id = os.path.splitext(fname)[0]
    row = {'full_path': os.path.join(ISIC_IMG, fname),
           'source': 'isic', 'diagnosis_bin': -1}  # нет диагноза в ISIC
    for feat in FEATURES:
        row[feat] = get_isic_label(img_id, feat)
    row['mask_path'] = build_combined_mask(img_id)
    records.append(row)
isic_df = pd.DataFrame(records)
print(f'\nISIC — {len(isic_df)} записей | масок: {isic_df["mask_path"].notna().sum()}')

# Train = ISIC + Derm7pt train
train_df = pd.concat([isic_df, derm_train], ignore_index=True)
print(f'\nTrain Эксп.6: {len(train_df)} '
      f'(ISIC={( train_df["source"]=="isic").sum()}, '
      f'Derm7pt={(train_df["source"]=="derm7pt").sum()})')

### 4. Dataset, 4 канала (RGB + маска)

Идентично стратегии A из Эксп.5, но возвращает три значения:  
(img_4ch, feat_labels_7, diag_label)

In [ ]:
class SkinDataset6(Dataset):
    """
    Dataset для Эксп.6 (многозадачная архитектура).
    Вход модели: 4 канала (RGB + сводная маска).
    Возвращает: (тензор 4ch, метки 7 признаков, метка диагноза).
    Для ISIC diagnosis_bin = -1, голова диагноза игнорирует этот пример.
    """
    def __init__(self, df, is_train=False, sz=300):
        self.df = df.reset_index(drop=True)
        self.is_train, self.sz = is_train, sz
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self): return len(self.df)

    def _clahe(self, arr):
        try:
            import cv2
            u8  = (arr * 255).astype(np.uint8)
            lab = cv2.cvtColor(u8, cv2.COLOR_RGB2LAB)
            cl  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            lab[:, :, 0] = cl.apply(lab[:, :, 0])
            return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.
        except: return arr

    def _quality_aug(self, img):
        if torch.rand(1) > 0.3: return img
        scale = torch.FloatTensor(1).uniform_(0.5, 0.9).item()
        small = max(64, int(self.sz * scale))
        return img.resize((small, small), Image.BILINEAR).resize(
            (self.sz, self.sz), Image.BILINEAR)

    def _load_mask(self, mask_path):
        if pd.notna(mask_path) and mask_path and os.path.exists(str(mask_path)):
            return np.array(
                Image.open(mask_path).convert('L').resize(
                    (self.sz, self.sz), Image.NEAREST),
                np.float32) / 255.
        return np.zeros((self.sz, self.sz), np.float32)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(row['full_path']).convert('RGB')
        if self.is_train: img = self._quality_aug(img)
        img  = img.resize((self.sz, self.sz), Image.BILINEAR)
        arr  = np.array(img, np.float32) / 255.
        if self.is_train and torch.rand(1) > 0.5: arr = self._clahe(arr)
        mask = self._load_mask(row.get('mask_path'))

        t4 = torch.from_numpy(
            np.concatenate([arr, mask[:, :, np.newaxis]], axis=2)
        ).permute(2, 0, 1)
        t4[:3] = (t4[:3] - self.mean) / self.std
        t4[3]  = (t4[3] - 0.5) / 0.5

        if self.is_train:
            if torch.rand(1) > .5: t4 = torch.flip(t4, [2])
            if torch.rand(1) > .5: t4 = torch.flip(t4, [1])

        feat_labels = torch.tensor(row[FEATURES].values.astype(np.float32))
        diag_label  = torch.tensor(float(row.get('diagnosis_bin', -1)))
        return t4, feat_labels, diag_label

### 5. Многозадачная архитектура: 8 независимых голов
  
7 отдельных голов для признаков + 1 голова для диагноза меланомы.

У Kawahara et al. (2018) Multi-task Inception-v4 с 8 ветвями.  
У нас EfficientNet-B3 с общим backbone и 8 независимыми головами.

In [ ]:
class MultiHeadEfficientNet(nn.Module):
    """
    EfficientNet-B3 с 8 независимыми классификационными головами.

    Архитектура вдохновлена Kawahara et al. (2018):
      Голова 0-6: по одной на каждый признак Argenziano (7-point checklist)
      Голова 7:   диагностический вердикт (меланома / не меланома)

    Backbone общий, неявный feature sharing между всеми головами.
    Каждая голова специализируется на своей задаче независимо.

    Отличие от Kawahara: у них отдельные backbone-ветви для каждой головы.
    У нас один backbone, что эффективнее при ограниченных данных.
    """
    def __init__(self, n_features=7, in_channels=4, dropout=0.3):
        super().__init__()

        # Backbone
        base     = timm.create_model('efficientnet_b3', pretrained=True,
                                     num_classes=0)
        feat_dim = base.num_features  # 1536

        # Расширяем первый conv 3→4 каналов
        old = base.conv_stem
        new = nn.Conv2d(in_channels, old.out_channels,
                        old.kernel_size, old.stride, old.padding,
                        bias=old.bias is not None)
        with torch.no_grad():
            new.weight[:, :3] = old.weight
            new.weight[:, 3:] = old.weight.mean(dim=1, keepdim=True)
        base.conv_stem   = new
        self.backbone    = base

        # 7 голов для признаков Argenziano
        self.feature_heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))
            for _ in range(n_features)
        ])

        # 8-я голова: диагноз (меланома / не меланома)
        self.diagnosis_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 1)
        )

        n_params = sum(p.numel() for p in self.parameters()) / 1e6
        print(f'MultiHeadEfficientNet-B3:')
        print(f'  Параметров: {n_params:.1f}M')
        print(f'  Backbone:   EfficientNet-B3 ({feat_dim}D features)')
        print(f'  Вход:       {in_channels} канала (RGB + маска)')
        print(f'  Головы:     {n_features} × признак + 1 × диагноз')

    def forward(self, x):
        """
        x: (B, 4, H, W)
        Возвращает:
          feat_logits: (B, 7) — логиты 7 признаков
          diag_logit:  (B, 1) — логит диагноза меланомы
        """
        f = self.backbone(x)  # (B, 1536)
        feat_logits = torch.cat(
            [h(f) for h in self.feature_heads], dim=1) # (B, 7)
        diag_logit  = self.diagnosis_head(f) # (B, 1)
        return feat_logits, diag_logit

### 6. Многозадачная функция потерь

Взвешенная сумма Focal Loss по всем 8 головам.  
Метки "−1" игнорируются (т.к нет аннотации в ISIC для части признаков и для диагноза).

In [ ]:
class MultiTaskFocalLoss(nn.Module):
    """
    Многозадачная Focal Loss для 8 голов.

    Для каждой головы i:
      label = -1, пропускаем (нет аннотации)
      label = 0/1, считаем Focal Loss с fn_weight

    Итог: взвешенная сумма по всем головам.
    HEAD_WEIGHTS[i] задаёт клиническую важность каждой головы.
    """
    def __init__(self, alpha=0.25, gamma=2.0, fn_weight=3.0,
                 head_weights=None):
        super().__init__()
        self.alpha  = alpha
        self.gamma  = gamma
        self.fn_w   = fn_weight
        hw = head_weights if head_weights else [1.0] * 8
        self.register_buffer('hw', torch.tensor(hw, dtype=torch.float32))

    def _focal_one(self, logits, targets):
        """Focal Loss для одной головы с маскированием -1."""
        mask  = (targets >= 0).float()
        tgt_c = targets.clamp(min=0)
        pw    = torch.ones_like(logits) + (self.fn_w - 1) * tgt_c
        bce   = nn.functional.binary_cross_entropy_with_logits(
                    logits, tgt_c, weight=pw, reduction='none')
        fl    = self.alpha * (1 - torch.exp(-bce)) ** self.gamma * bce
        n     = mask.sum()
        return (fl * mask).sum() / n if n > 0 else fl.sum() * 0

    def forward(self, feat_logits, diag_logit, feat_labels, diag_label):
        total = torch.tensor(0.0, device=feat_logits.device)
        for i in range(feat_logits.shape[1]):
            total = total + self.hw[i] * self._focal_one(
                feat_logits[:, i], feat_labels[:, i])
        total = total + self.hw[7] * self._focal_one(
            diag_logit.squeeze(1), diag_label)
        return total

### 7. Функции обучения и оценки

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train(); total = 0
    for x, feat_labels, diag_label in tqdm(loader, desc='train', leave=False):
        x, feat_labels, diag_label = (
            x.to(device), feat_labels.to(device), diag_label.to(device))
        optimizer.zero_grad()
        feat_logits, diag_logit = model(x)
        loss = criterion(feat_logits, diag_logit, feat_labels, diag_label)
        loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total = 0
    all_fp, all_pp, all_fl = [], [], []   # feat probs/preds/labels
    all_dp, all_dl         = [], []       # diag probs/labels

    for x, feat_labels, diag_label in tqdm(loader, desc='eval', leave=False):
        x, feat_labels, diag_label = (
            x.to(device), feat_labels.to(device), diag_label.to(device))
        feat_logits, diag_logit = model(x)
        total += criterion(feat_logits, diag_logit, feat_labels, diag_label).item()

        fp = torch.sigmoid(feat_logits).cpu().numpy()
        dp = torch.sigmoid(diag_logit).squeeze(1).cpu().numpy()
        all_fp.append(fp)
        all_pp.append((fp >= THRESHOLD).astype(int))
        all_fl.append(feat_labels.cpu().numpy())
        all_dp.append(dp)
        all_dl.append(diag_label.cpu().numpy())

    feat_probs  = np.vstack(all_fp)
    feat_preds  = np.vstack(all_pp)
    feat_labels = np.vstack(all_fl)
    diag_probs  = np.concatenate(all_dp)
    diag_labels = np.concatenate(all_dl)

    # Метрики признаков
    f1s, aucs = [], []
    for i in range(feat_labels.shape[1]):
        m = feat_labels[:, i] >= 0
        if not m.any(): continue
        f1s.append(f1_score(feat_labels[m, i], feat_preds[m, i], zero_division=0))
        try:    aucs.append(roc_auc_score(feat_labels[m, i], feat_probs[m, i]))
        except: aucs.append(0.)

    # Метрики головы диагноза
    dm = diag_labels >= 0
    diag_metrics = {}
    if dm.sum() > 0:
        diag_preds_bin = (diag_probs[dm] >= THRESHOLD).astype(int)
        try:
            diag_metrics['auc'] = roc_auc_score(diag_labels[dm], diag_probs[dm])
        except: diag_metrics['auc'] = 0.
        diag_metrics['f1']     = f1_score(diag_labels[dm], diag_preds_bin, zero_division=0)
        diag_metrics['recall'] = recall_score(diag_labels[dm], diag_preds_bin, zero_division=0)
        diag_metrics['n']      = int(dm.sum())

    return {
        'loss':         total / len(loader),
        'macro_f1':     float(np.mean(f1s)),
        'macro_auc':    float(np.mean(aucs)),
        'feat_probs':   feat_probs,
        'feat_preds':   feat_preds,
        'feat_labels':  feat_labels,
        'diag_probs':   diag_probs,
        'diag_labels':  diag_labels,
        'diag_metrics': diag_metrics,
    }

### 8. Подсчёт баллов по шкале Argenziano

$S = \sum_{i=1}^{7} w_i \cdot \hat{y}_i$, где $w_i$ — вес признака (2 или 1 балл)

При $S \geq 3$ подозрение на меланому

In [ ]:
def compute_argenziano_score(feat_probs, threshold=THRESHOLD):
    preds = (feat_probs >= threshold).astype(int)
    score = int(np.dot(preds, ARGENZIANO_SCORES))
    found = [FEAT_RU[i] for i, p in enumerate(preds) if p == 1]
    return score, int(score >= SUSPICION_THRESHOLD), found


def argenziano_analysis(m, name=''):
    probs  = m['feat_probs']
    labels = m['feat_labels']
    N      = len(probs)

    scores   = np.array([compute_argenziano_score(probs[i])[0] for i in range(N)])
    verdicts = (scores >= SUSPICION_THRESHOLD).astype(int)

    gt_scores = np.array([
        sum(ARGENZIANO_WEIGHTS[f] * int(labels[i, j] == 1)
            for j, f in enumerate(FEATURES) if labels[i, j] >= 0)
        for i in range(N)
    ])
    gt_verdicts = (gt_scores >= SUSPICION_THRESHOLD).astype(int)

    cm = confusion_matrix(gt_verdicts, verdicts)
    print(f'\n  [{name}] Argenziano score:')
    print(f'  Подозрительных: {verdicts.sum()} ({100*verdicts.mean():.1f}%)')
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f'  Чувствительность: {sens:.3f} | Специфичность: {spec:.3f}')
    return scores, verdicts

### 9. Запуск эксперимента 6

12 эпох, 3007 изображений

In [ ]:
# Датасеты
train_dataset = SkinDataset6(train_df, is_train=True)
val_dataset   = SkinDataset6(val_df,   is_train=False)
test_dataset  = SkinDataset6(test_df,  is_train=False)

# Модель и лосс
model6    = MultiHeadEfficientNet(n_features=7, in_channels=4, dropout=0.3).to(device)
criterion = MultiTaskFocalLoss(
    alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
    fn_weight=FN_WEIGHT, head_weights=HEAD_WEIGHTS)

# DataLoaders
ld_tr = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                   num_workers=2, drop_last=True)
ld_vl = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                   num_workers=2)
ld_te = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                   num_workers=2)

optimizer = optim.AdamW(model6.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

SAVE_PATH = '/content/best_exp6.pth'
best_loss = float('inf')
tl_hist, vl_hist, vf_hist, va_hist = [], [], [], []

print(f'\n{"═"*62}')
print(f'  ЭКСПЕРИМЕНТ 6: MultiHead EfficientNet-B3 (8 голов)')
print(f'  Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'  HEAD_WEIGHTS: {HEAD_WEIGHTS}')
print(f'  fn_weight={FN_WEIGHT} | threshold={THRESHOLD}')
print(f'{"═"*62}')

for ep in range(1, NUM_EPOCHS + 1):
    tl = train_epoch(model6, ld_tr, optimizer, criterion)
    vm = evaluate(model6, ld_vl, criterion)
    scheduler.step()

    tl_hist.append(tl)
    vl_hist.append(vm['loss'])
    vf_hist.append(vm['macro_f1'])
    va_hist.append(vm['macro_auc'])

    dm = vm['diag_metrics']
    diag_str = ''
    if dm:
        diag_str = (f" | DiagAUC={dm.get('auc',0):.3f} "
                    f"F1={dm.get('f1',0):.3f} "
                    f"Rec={dm.get('recall',0):.3f} (n={dm.get('n',0)})")

    print(f'[Эксп.6] ep {ep:02d}/{NUM_EPOCHS} | '
          f'train={tl:.4f}  val={vm["loss"]:.4f}  '
          f'F1={vm["macro_f1"]:.3f}  AUC={vm["macro_auc"]:.3f}'
          f'{diag_str}')

    if vm['loss'] < best_loss:
        best_loss = vm['loss']
        torch.save(model6.state_dict(), SAVE_PATH)
        print(f'  ✓ Лучшая модель сохранена (ep {ep})')

# Загружаем лучшую модель
model6.load_state_dict(torch.load(SAVE_PATH))
print('\n✓ Лучшая модель загружена')

### 10. Кривые обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Кривые обучения — Эксп.6 (MultiHead 8 голов)', fontsize=13)

best_ep = int(np.argmin(vl_hist))
axes[0].plot(tl_hist, label='Train Loss', marker='o', ms=3, color='#4C72B0')
axes[0].plot(vl_hist, label='Val Loss',   marker='s', ms=3, ls='--', color='#C44E52')
axes[0].axvline(best_ep, color='gray', ls=':', alpha=.7, label=f'Best ep {best_ep+1}')
axes[0].set_xlabel('Эпоха'); axes[0].set_title('Focal Loss (многозадачная)')
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(vf_hist, color='#55A868', marker='o', ms=3, label='Macro F1')
axes[1].plot(va_hist, color='#4C72B0', marker='s', ms=3, ls='--', label='Macro AUC')
axes[1].axhline(0.5,   ls='--', color='red',    alpha=.5, label='Baseline 0.5')
axes[1].axhline(0.531, ls=':',  color='orange', alpha=.7, label='Эксп.5A (0.531)')
axes[1].set_xlabel('Эпоха'); axes[1].set_title('Метрики на валидации')
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig('/content/exp6_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 11. Метрики на валидации и тесте

In [ ]:
def print_metrics(m, title=''):
    print(f'\n{"="*62}')
    print(f'  {title}')
    print(f'  Macro F1: {m["macro_f1"]:.4f}  |  Macro AUC: {m["macro_auc"]:.4f}')
    print(f'{"─"*62}')
    print(f'  {"Признак":<26} {"F1":>6} {"Prec":>6} {"Recall":>7} '
          f'{"AUC":>7} {"Pos":>5} {"Балл":>6}')
    print(f'{"─"*62}')
    for i, (feat, name) in enumerate(zip(FEATURES, FEAT_RU)):
        msk = m['feat_labels'][:, i] >= 0
        if not msk.any(): continue
        f1  = f1_score(m['feat_labels'][msk,i], m['feat_preds'][msk,i], zero_division=0)
        pr  = precision_score(m['feat_labels'][msk,i], m['feat_preds'][msk,i], zero_division=0)
        rec = recall_score(m['feat_labels'][msk,i], m['feat_preds'][msk,i], zero_division=0)
        try:    auc = roc_auc_score(m['feat_labels'][msk,i], m['feat_probs'][msk,i])
        except: auc = 0.
        pos  = int(m['feat_labels'][msk,i].sum())
        wt   = ARGENZIANO_WEIGHTS[feat]
        star = ' ★' if wt == 2 else ''
        print(f'  {name:<26} {f1:>6.3f} {pr:>6.3f} {rec:>7.3f} '
              f'{auc:>7.3f} {pos:>5}  {wt}б{star}')
    print(f'{"─"*62}')
    print(f'  {"MACRO AVG":<26} {m["macro_f1"]:>6.3f} {"":>6} '
          f'{"":>7} {m["macro_auc"]:>7.3f}')

    dm = m.get('diag_metrics', {})
    if dm:
        print(f'\n  {"─"*56}')
        print(f'  ГОЛОВА 8 — ДИАГНОЗ (меланома / не меланома)')
        print(f'  Примеров с аннотацией: {dm.get("n",0)} (только Derm7pt)')
        print(f'  AUC-ROC: {dm.get("auc",0):.4f}  |  '
              f'F1: {dm.get("f1",0):.4f}  |  Recall: {dm.get("recall",0):.4f}')
    print(f'{"="*62}')

# Val
val_metrics  = evaluate(model6, ld_vl, criterion)
print_metrics(val_metrics,  'ЭКСП.6 — Val = Derm7pt (203 изображения)')

# Test
test_metrics = evaluate(model6, ld_te, criterion)
print_metrics(test_metrics, 'ЭКСП.6 — Test = Derm7pt (395 изображений)')

### 12. Confusion Matrices (Val)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Confusion Matrices — Эксп.6 MultiHead (Val = Derm7pt)', fontsize=13)

for i, (feat, name) in enumerate(zip(FEATURES, FEAT_RU)):
    ax   = axes[i // 4][i % 4]
    msk  = val_metrics['feat_labels'][:, i] >= 0
    if not msk.any(): ax.set_visible(False); continue
    cm_  = confusion_matrix(val_metrics['feat_labels'][msk, i],
                             val_metrics['feat_preds'][msk, i])
    f1_  = f1_score(val_metrics['feat_labels'][msk, i],
                    val_metrics['feat_preds'][msk, i], zero_division=0)
    rec_ = recall_score(val_metrics['feat_labels'][msk, i],
                        val_metrics['feat_preds'][msk, i], zero_division=0)
    sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Норма', 'Патология'],
                yticklabels=['Норма', 'Патология'], cbar=False)
    wt = ARGENZIANO_WEIGHTS[feat]
    ax.set_title(f'{name} ({wt}б{" ★" if wt==2 else ""})\n'
                 f'F1={f1_:.2f}  Recall={rec_:.2f}', fontsize=9)
    ax.set_xlabel('Предсказание'); ax.set_ylabel('Истина')

axes[1][3].set_visible(False)
plt.tight_layout()
plt.savefig('/content/exp6_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

### 13. Анализ баллов Argenziano

In [ ]:
scores6, verdicts6 = argenziano_analysis(val_metrics, 'Эксп.6')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Анализ по шкале Argenziano — Эксп.6', fontsize=13)

bins = np.arange(-0.5, 11.5, 1)
axes[0].hist(scores6, bins=bins, color='#4C72B0', alpha=0.85, edgecolor='white')
axes[0].axvline(SUSPICION_THRESHOLD - 0.5, color='red', ls='--', lw=2,
                label=f'Порог подозрения ({SUSPICION_THRESHOLD}б)')
axes[0].set_xlabel('Балл по шкале Argenziano')
axes[0].set_ylabel('Количество изображений')
axes[0].set_title('Распределение баллов (Val)')
axes[0].set_xticks(range(0, 11))
axes[0].legend(); axes[0].grid(alpha=.3)

feat_contrib = [(val_metrics['feat_preds'][:, i].sum() * ARGENZIANO_SCORES[i])
                for i in range(len(FEATURES))]
colors_b = ['#C44E52' if ARGENZIANO_WEIGHTS[f]==2 else '#4C72B0' for f in FEATURES]
bars = axes[1].bar(FEAT_RU, feat_contrib, color=colors_b, alpha=0.85)
for bar, v in zip(bars, feat_contrib):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(v), ha='center', va='bottom', fontsize=9)
axes[1].set_xticklabels(FEAT_RU, rotation=30, ha='right', fontsize=9)
axes[1].set_title('Суммарный вклад признаков в баллы\n'
                  '(красный = 2б ★, синий = 1б)')
axes[1].set_ylabel('Суммарный балл по выборке')
axes[1].grid(axis='y', alpha=.3)

plt.tight_layout()
plt.savefig('/content/exp6_argenziano.png', dpi=150, bbox_inches='tight')
plt.show()

### 14. Grad-CAM

Строим Grad-CAM + баллы Argenziano + сравнение с GT диагнозом (ответы врачей)

In [ ]:
"""
Для каждого примера показываем:
  - Оригинальное изображение
  - Карты Grad-CAM для каждого из 7 признаков
  - Наши предсказания (вероятности + бинарные метки)
  - Реальные метки специалистов (GT)
  - Балл Argenziano: наш vs GT
  - Итоговый вердикт: наш vs реальный диагноз
"""

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import cv2

# Grad-CAM для EfficientNet-B3

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_full_backward_hook(backward_hook)

    def generate(self, x, class_idx):
        self.model.zero_grad()
        feat_logits, diag_logit = self.model(x)

        # Выбираем нужную голову
        if class_idx < 7:
            score = feat_logits[0, class_idx]
        else:
            score = diag_logit[0, 0]

        score.backward()

        # Взвешиваем карты активации
        weights   = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam       = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam       = F.relu(cam)
        cam       = cam.squeeze().cpu().numpy()

        # Нормализуем
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


def apply_colormap(cam, img_np, alpha=0.5):
    """Накладываем тепловую карту на изображение."""
    h, w = img_np.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(
        (cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * img_np * 255).astype(np.uint8)
    return overlay


# Получаем целевой слой backbone для Grad-CAM
target_layer = model6.backbone.blocks[-1] # последний блок EfficientNet

gradcam = GradCAM(model6, target_layer)
model6.eval()

# Функция визуализации одного примера

def visualize_example(idx, df, dataset, title_prefix='Val'):
    """
    Полная визуализация для одного изображения:
    оригинал + 7 Grad-CAM карт + сводка предсказаний vs GT.
    """
    row = df.iloc[idx]

    # Загружаем тензор
    item      = dataset[idx]
    x         = item[0].unsqueeze(0).to(device) # (1, 4, H, W)
    gt_feats  = item[1].numpy() # (7,)
    gt_diag   = item[2].item() # 0/1/-1

    # Получаем предсказания
    with torch.no_grad():
        feat_logits, diag_logit = model6(x)
    feat_probs = torch.sigmoid(feat_logits).cpu().numpy()[0]  # (7,)
    feat_preds = (feat_probs >= THRESHOLD).astype(int)
    diag_prob  = torch.sigmoid(diag_logit).cpu().item()

    # Баллы Argenziano
    our_score, our_verdict, our_found  = compute_argenziano_score(feat_probs)
    gt_score = sum(
        ARGENZIANO_WEIGHTS[f] * int(gt_feats[j] == 1)
        for j, f in enumerate(FEATURES) if gt_feats[j] >= 0
    )
    gt_verdict = int(gt_score >= SUSPICION_THRESHOLD)

    # Оригинальное изображение (RGB без нормализации)
    img_pil = Image.open(row['full_path']).convert('RGB').resize((300, 300))
    img_np  = np.array(img_pil)

    # Строим фигуру
    fig = plt.figure(figsize=(22, 9))
    fig.patch.set_facecolor('#1a1a2e')

    # Сетка: 2 строки × 9 колонок
    # Колонка 0 - оригинал (двойная высота)
    # Колонки 1-7 — Grad-CAM карты
    # Колонка 8 — сводная таблица
    gs = fig.add_gridspec(2, 9, hspace=0.3, wspace=0.25)

    # Оригинал
    ax_orig = fig.add_subplot(gs[:, 0])
    ax_orig.imshow(img_np)
    ax_orig.axis('off')

    # Заголовок с диагнозом
    real_diag = row.get('diagnosis', '?')
    our_diag_str  = f'Меланома ({diag_prob:.2f})' if diag_prob >= THRESHOLD else f'Не меланома ({diag_prob:.2f})'
    color_our = '#ff4444' if diag_prob >= THRESHOLD else '#44ff88'
    color_gt  = '#ff4444' if 'melanoma' in str(real_diag).lower() else '#44ff88'

    ax_orig.set_title(
        f'{title_prefix} #{idx}\n'
        f'GT диагноз: {real_diag}\n'
        f'Наш вердикт: {our_diag_str}',
        fontsize=8, color='white', pad=6,
        fontweight='bold'
    )

    # Grad-CAM карты (2 строки × 4 колонки, 7 признаков)
    positions = [(0,1),(0,2),(0,3),(0,4),(1,1),(1,2),(1,3)]
    for feat_idx, (row_pos, col_pos) in enumerate(positions):
        ax = fig.add_subplot(gs[row_pos, col_pos])

        # Генерируем Grad-CAM
        x_grad = x.clone().requires_grad_(True)
        cam    = gradcam.generate(x_grad, feat_idx)
        overlay = apply_colormap(cam, img_np / 255., alpha=0.55)

        ax.imshow(overlay)
        ax.axis('off')

        feat_name = FEAT_RU[feat_idx]
        prob      = feat_probs[feat_idx]
        pred      = feat_preds[feat_idx]
        gt        = int(gt_feats[feat_idx]) if gt_feats[feat_idx] >= 0 else -1
        wt        = ARGENZIANO_SCORES[feat_idx]

        # Цвет заголовка: зелёный = pred==gt, красный = ошибка
        if gt == -1:
            title_color = 'gray'
            match_str   = 'нет GT'
        elif pred == gt:
            title_color = '#44ff88'
            match_str   = '✓'
        else:
            title_color = '#ff6666'
            match_str   = '✗'

        ax.set_title(
            f'{feat_name} ({wt}б)\n'
            f'P={prob:.2f}  Pred={pred}  GT={gt if gt>=0 else "?"} {match_str}',
            fontsize=7.5, color=title_color, pad=3
        )

    # Сводная таблица (правый столбец)
    ax_tbl = fig.add_subplot(gs[:, 8])
    ax_tbl.set_facecolor('#0d0d1a')
    ax_tbl.axis('off')

    lines = [
        ('НАШИ ПРЕДСКАЗАНИЯ', 'white', 11, True),
        ('', 'white', 8, False),
    ]
    for j, feat in enumerate(FEATURES):
        prob = feat_probs[j]
        pred = feat_preds[j]
        gt   = int(gt_feats[j]) if gt_feats[j] >= 0 else -1
        wt   = ARGENZIANO_SCORES[j]
        name = FEAT_RU[j]

        if gt == -1:
            c = 'gray'
        elif pred == gt:
            c = '#44ff88'
        else:
            c = '#ff6666'

        lines.append((
            f'{name} ({wt}б): {prob:.2f} → {"+" if pred else "−"}  GT={"+" if gt==1 else ("−" if gt==0 else "?")}',
            c, 8, False
        ))

    lines += [
        ('', 'white', 8, False),
        ('─' * 30, '#888888', 7, False),
        (f'Наш балл Argenziano: {our_score}', '#ffdd44', 10, True),
        (f'GT балл Argenziano:  {gt_score}', '#aaaaaa', 10, False),
        ('', 'white', 8, False),
    ]

    our_v_str = '⚠ ПОДОЗРЕНИЕ' if our_verdict else '✓ НОРМА'
    gt_v_str  = '⚠ ПОДОЗРЕНИЕ' if gt_verdict  else '✓ НОРМА'
    lines += [
        (f'Наш вердикт:  {our_v_str}',
         '#ff4444' if our_verdict else '#44ff88', 9, True),
        (f'GT вердикт:   {gt_v_str}',
         '#ff4444' if gt_verdict  else '#44ff88', 9, False),
        ('', 'white', 8, False),
        ('─' * 30, '#888888', 7, False),
        (f'Диагноз голова 8:', 'white', 8, True),
        (f'{our_diag_str}', color_our, 9, True),
        (f'GT диагноз: {real_diag}', color_gt, 8, False),
    ]

    y = 0.98
    for text, color, size, bold in lines:
        ax_tbl.text(
            0.02, y, text,
            transform=ax_tbl.transAxes,
            fontsize=size, color=color,
            fontweight='bold' if bold else 'normal',
            verticalalignment='top',
            fontfamily='monospace'
        )
        y -= 0.055 if size >= 9 else 0.042

    fig.suptitle(
        f'Grad-CAM + Argenziano + GT сравнение — Эксп.6 | {title_prefix} #{idx}',
        color='white', fontsize=11, y=1.01
    )

    plt.savefig(f'/content/gradcam_example_{title_prefix}_{idx}.png',
                dpi=130, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    plt.close()


# Выбираем интересные примеры для показа

def find_interesting_examples(df, dataset, n_per_group=2):
    """
    Ищем разнообразные примеры:
    - Правильно классифицированные меланомы
    - Правильно классифицированные не меланомы
    - Ошибочно классифицированные (FP и FN)
    """
    model6.eval()
    results = []

    for idx in range(len(dataset)):
        item     = dataset[idx]
        x        = item[0].unsqueeze(0).to(device)
        gt_feats = item[1].numpy()
        gt_diag  = item[2].item()

        with torch.no_grad():
            feat_logits, diag_logit = model6(x)
        feat_probs = torch.sigmoid(feat_logits).cpu().numpy()[0]
        diag_prob  = torch.sigmoid(diag_logit).cpu().item()

        our_score, our_verdict, _ = compute_argenziano_score(feat_probs)
        gt_score = sum(
            ARGENZIANO_WEIGHTS[f] * int(gt_feats[j] == 1)
            for j, f in enumerate(FEATURES) if gt_feats[j] >= 0
        )
        gt_verdict = int(gt_score >= SUSPICION_THRESHOLD)

        real_diag = df.iloc[idx].get('diagnosis', '')
        is_melanoma_gt = 'melanoma' in str(real_diag).lower()

        results.append({
            'idx':           idx,
            'our_score':     our_score,
            'gt_score':      gt_score,
            'our_verdict':   our_verdict,
            'gt_verdict':    gt_verdict,
            'diag_prob':     diag_prob,
            'is_mel_gt':     is_melanoma_gt,
            'score_diff':    abs(our_score - gt_score),
        })

    import pandas as pd
    res_df = pd.DataFrame(results)

    chosen = []

    # 1. Правильные меланомы (our_verdict=1, gt_verdict=1, is_mel_gt=True)
    tp_mel = res_df[(res_df.our_verdict==1) & (res_df.gt_verdict==1) &
                    res_df.is_mel_gt].nlargest(n_per_group, 'diag_prob')
    chosen += list(tp_mel['idx'])

    # 2. Правильные не меланомы (our_verdict=0, gt_verdict=0)
    tn = res_df[(res_df.our_verdict==0) & (res_df.gt_verdict==0) &
                ~res_df.is_mel_gt].nsmallest(n_per_group, 'our_score')
    chosen += list(tn['idx'])

    # 3. Ошибка: пропустили подозрение (FN: our=0, gt=1)
    fn = res_df[(res_df.our_verdict==0) & (res_df.gt_verdict==1)].head(1)
    chosen += list(fn['idx'])

    # 4. Ошибка: лишняя тревога (FP: our=1, gt=0)
    fp = res_df[(res_df.our_verdict==1) & (res_df.gt_verdict==0) &
                ~res_df.is_mel_gt].nsmallest(1, 'gt_score')
    chosen += list(fp['idx'])

    return list(dict.fromkeys(chosen)) # убираем дубли, сохраняем порядок


# Запускаем визуализацию

print('Ищем интересные примеры...')
interesting = find_interesting_examples(val_df, val_dataset, n_per_group=2)
print(f'Выбрано примеров: {len(interesting)}: {interesting}')

print('\nГенерируем Grad-CAM визуализации...')
for idx in interesting:
    print(f'  Пример {idx}...')
    visualize_example(idx, val_df, val_dataset, title_prefix='Val')

# Сохраняем на Drive
import shutil
for idx in interesting:
    fname = f'gradcam_example_Val_{idx}.png'
    src   = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, fname))
        print(f'✓ {fname}')

print('\n✓ Grad-CAM анализ завершён!')

### 15. Финальная сводная таблица всех экспериментов

Сравниваем все эксперименты, включая результаты Kawahara et al. (2018)

In [ ]:
all_results = [
    ('Эксп.1  Derm7pt, 3ch, EfficientNet-B3',         0.499, 0.703),
    ('Эксп.2  ISIC, 3ch',                              0.391, 0.533),
    ('Эксп.3  ISIC, 4ch+маска',                        0.385, 0.538),
    ('Эксп.4  Derm+ISIC, 4ch+CLAHE+aug',               0.514, 0.737),
    ('Эксп.5A Derm+ISIC, 4ch, fn_weight=3',            0.531, 0.740),
    ('Эксп.5B Derm+ISIC, masked input',                0.471, 0.607),
    ('Эксп.5C Derm+ISIC, dual-stream',                 0.507, 0.779),
    ('Эксп.6  MultiHead 8 голов (наш лучший)',
     val_metrics['macro_f1'], val_metrics['macro_auc']),
]

baseline = 0.499
print(f'\n{"="*70}')
print(f'  ФИНАЛЬНАЯ СВОДНАЯ ТАБЛИЦА — ВСЕ ЭКСПЕРИМЕНТЫ')
print(f'  Val = Derm7pt (203 изображения), threshold = {THRESHOLD}')
print(f'{"="*70}')
print(f'  {"Конфигурация":<52} {"F1":>6} {"AUC":>7} {"ΔF1":>7}')
print(f'  {"─"*67}')
for name, f1, auc in all_results:
    delta = f1 - baseline
    sign  = '+' if delta >= 0 else ''
    best  = ' ←' if name.startswith('Эксп.6') else ''
    print(f'  {name:<52} {f1:>6.3f} {auc:>7.3f} {sign}{delta:>6.3f}{best}')
print(f'  {"─"*67}')
print(f'  {"Kawahara et al. (2018) — Multi-task Inception-v4":<52} '
      f'  —     0.838      —')
print(f'{"="*70}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Финальное сравнение всех экспериментов (Val = Derm7pt)', fontsize=13)

names = [r[0].split('  ')[0] for r in all_results]
f1s   = [r[1] for r in all_results]
aucs  = [r[2] for r in all_results]
colors = ['#4C72B0','#aec7e8','#aec7e8','#55A868',
          '#C44E52','#ffbb78','#8c564b','#d62728']

for ax, vals, title, yl in [
    (axes[0], f1s,  'Macro F1',      'F1'),
    (axes[1], aucs, 'Macro AUC-ROC', 'AUC'),
]:
    bars = ax.bar(names, vals, color=colors, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
    ax.axhline(0.5, ls='--', color='gray', alpha=.5, label='Baseline 0.5')
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel(yl); ax.set_title(title)
    ax.legend(); ax.grid(axis='y', alpha=.3)
    ax.set_ylim(0.3, 0.85)

plt.tight_layout()
plt.savefig('/content/exp6_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 16. Сохраняем на Google Drive

In [ ]:
files = [
    'best_exp6.pth',
    'exp6_curves.png',
    'exp6_confusion.png',
    'exp6_argenziano.png',
    'exp6_final_comparison.png',
]
print('Сохраняем на Google Drive...')
for f in files:
    src = f'/content/{f}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, f))
        print(f'  ✓ {f}')
    else:
        print(f'  ✗ не найдено: {f}')

# JSON с результатами
results = {
    'val':  {'macro_f1': round(val_metrics['macro_f1'],4),
             'macro_auc': round(val_metrics['macro_auc'],4),
             'diag_auc':  round(val_metrics['diag_metrics'].get('auc',0),4),
             'diag_f1':   round(val_metrics['diag_metrics'].get('f1',0),4),
             'diag_recall': round(val_metrics['diag_metrics'].get('recall',0),4)},
    'test': {'macro_f1': round(test_metrics['macro_f1'],4),
             'macro_auc': round(test_metrics['macro_auc'],4)},
    'argenziano': {'n_suspicious': int(verdicts6.sum()),
                   'pct_suspicious': round(100*verdicts6.mean(),1)},
}
with open(os.path.join(SAVE_DIR, 'results_exp6.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('  ✓ results_exp6.json')
print(f'\n✓ Эксперимент 6 завершён!')
print(f'  Val  F1={val_metrics["macro_f1"]:.4f}  AUC={val_metrics["macro_auc"]:.4f}')
print(f'  Test F1={test_metrics["macro_f1"]:.4f}  AUC={test_metrics["macro_auc"]:.4f}')